# 04 · Je Eigen Virtuele Sensor Bouwen: ESP32 + DHT22 in Wokwi

In de vorige notebooks hebben we data verwerkt van gesimuleerde en fysieke sensoren. In deze afsluitende les ga je **zelf een virtueel IoT-apparaat bouwen en programmeren** — volledig in je webbrowser, zonder dat je fysieke hardware of soldeerbouten nodig hebt!

We gebruiken hiervoor de populaire online simulator **Wokwi** (<https://wokwi.com>). Wokwi simuleert een **ESP32 microcontroller** die gekoppeld is aan een **DHT22 temperatuur- en luchtvochtigheidssensor**. 

De Wokwi-simulator bevat een gesimuleerd Wi-Fi netwerk met *echte internettoegang*. Dit betekent dat jouw virtuele ESP32 **daadwerkelijk live berichten gaat publiceren** naar de openbare MQTT-broker (`broker.hivemq.com`), waarna iedereen in de klas jouw virtuele sensor live ziet binnenkomen in notebook 02!

---

### Wikipedia-stijl uitleg: Wat is een ESP32 en MicroPython?

1. **Microcontroller (ESP32):**
   * Een **microcontroller** is een complete 'computer op één enkele chip' (System-on-a-Chip / SoC). 
   * De **ESP32** (ontwikkeld door *Espressif Systems*) is de meest gebruikte microcontroller in moderne IoT-projecten. Hij beschikt over een snelle processor, ingebouwde **Wi-Fi** en **Bluetooth (BLE)**, en kost slechts zo'n € 5 tot € 8.
   * **GPIO Pins (General Purpose Input/Output):** Dit zijn de metalen pinnen aan de buitenkant van de chip waarmee de ESP32 elektronische componenten, sensoren en motoren kan aansturen of uitlezen.

2. **MicroPython:**
   * Traditiegetrouw worden microcontrollers geprogrammeerd in de ingewikkelde programmeertaal C++. 
   * **MicroPython** is een gestroomlijnde, efficiënte herimplementatie van de programmeertaal Python 3, speciaal geoptimaliseerd om rechtstreeks te draaien op de beperkte geheugencapaciteit van een microcontroller.

---

### Stappenplan: Project aanmaken in Wokwi (± 5 minuten)

1. Surf naar <https://wokwi.com> en klik rechtsboven op **+ New Project**. *(Je kunt zonder account direct simuleren; om je project op te slaan heb je een gratis account nodig).*
2. Kies onder het kopje *MicroPython* voor **MicroPython on ESP32**.
3. Je ziet in de Wokwi-interface twee tabbladen:
   * `main.py`: De MicroPython broncode.
   * `diagram.json`: Het elektrische aansluitschema van de onderdelen.
4. Vervang de inhoud van **beide tabbladen** door de code die in de volgende cellen wordt weergegeven (deze bestanden staan ook in de map `wokwi/` van deze repository).
5. Pas in `main.py` de variabelen `KLASCODE` en `AFDELING` aan.
6. Klik op de groene **▶ Start**-knop om de simulatie te starten!
7. **Simuleer een klimaatverandering:** Klik tijdens het draaien van de simulatie op de blauwe **DHT22-sensor** in het schema. Er verschijnt een schuifbalk waarmee je de temperatuur handmatig boven de 30 °C kunt schuiven. Je ziet meteen dat het bericht het alarm `"interrupt": 1` inschakelt!

In [1]:
# Inhoud van het elektrische aansluitschema: wokwi/diagram.json
# (Kopieer en plak onderstaande JSON-structuur in het tabblad 'diagram.json' van Wokwi)

import pathlib

pad_diagram = pathlib.Path('wokwi/diagram.json') if pathlib.Path('wokwi/diagram.json').exists() else pathlib.Path('../wokwi/diagram.json')
print(pad_diagram.read_text())

{
  "version": 1,
  "author": "I700301 Precisielandbouw - Starten met IoT",
  "editor": "wokwi",
  "parts": [
    { "type": "board-esp32-devkit-c-v4", "id": "esp", "top": 0, "left": 0, "attrs": {} },
    { "type": "wokwi-dht22", "id": "dht1", "top": -70, "left": 230, "attrs": { "temperature": "22.5", "humidity": "65" } }
  ],
  "connections": [
    [ "dht1:VCC", "esp:3V3", "red", [ "v0" ] ],
    [ "dht1:GND", "esp:GND.1", "black", [ "v0" ] ],
    [ "dht1:SDA", "esp:15", "green", [ "v0" ] ],
    [ "esp:TX", "$serialMonitor:RX", "", [] ],
    [ "esp:RX", "$serialMonitor:TX", "", [] ]
  ]
}



## 3. De MicroPython Broncode van het Toestel (`main.py`)

Bestudeer onderstaande code van boven naar beneden. Deze code volgt exact de **drie universele stappen** van vrijwel elk IoT-apparaat ter wereld:
1. **Verbinding maken met het netwerk:** De ESP32 verbindt met het draadloze Wi-Fi netwerk (*Wokwi-GUEST*) en synchroniseert zijn interne klok via een **NTP-tijdserver** (Network Time Protocol).
2. **Sensor uitlezen:** De microcontroller leest via **GPIO pin 15** de binaire datastroom van de DHT22-sensor in.
3. **Data verzenden via MQTT:** De data wordt omgezorgd naar een JSON-structuur en gepubliceerd naar de centrale MQTT-broker. Daarna pauzeert het apparaat tot de volgende meting.

In [2]:
# Inhoud van het hoofdprogramma: wokwi/main.py
# (Kopieer en plak onderstaande MicroPython code in het tabblad 'main.py' van Wokwi)

pad_main = pathlib.Path('wokwi/main.py') if pathlib.Path('wokwi/main.py').exists() else pathlib.Path('../wokwi/main.py')
print(pad_main.read_text())

# Virtuele stalsensor: ESP32 + DHT22 in de Wokwi-simulator, publiceert via MQTT
# in een JSON-formaat dat lijkt op dat van de Dragino S31-CB (zie notebook 01, stap 4).
#
# Pas enkel de drie regels onder INSTELLINGEN aan.

import network, time, json, machine, dht
from umqtt.simple import MQTTClient

# ---------- INSTELLINGEN ----------
KLASCODE  = "i700301-2026-xyz"      # dezelfde code als in de notebooks (docent geeft ze)
AFDELING  = "wokwi-groep01"         # jouw groepsnaam, uniek in de klas
INTERVAL  = 5                       # seconden tussen twee uplinks (echt toestel: 600)
# ----------------------------------

BROKER = "broker.hivemq.com"
TOPIC  = "ugent/{}/stal/{}".format(KLASCODE, AFDELING)
IMEI   = "8685080656" + "{:05d}".format(sum(ord(c) for c in AFDELING) % 100000)

# 1. Wifi: Wokwi simuleert een open netwerk "Wokwi-GUEST" met internettoegang
print("Verbinden met wifi", end="")
wlan = network.WLAN(network.STA_IF)
wlan.active(True)
wlan.connect("Wokwi-GUEST", "")
while not w

## 4. Live meekijken vanuit dit Python Notebook

Wanneer je de simulatie in Wokwi laat draaien, verstuurt jouw virtuele ESP32 live berichten over het echte internet! 

Met de onderstaande code kun je vanuit dit notebook luisteren naar jouw eigen Wokwi-sensor op het topic `ugent/<klascode>/stal/wokwi-+`.

In [3]:
import os    # Voor omgevingsvariabelen
import time  # Voor tijdsverloop en pauzes
import json  # Voor het ontleden van JSON-berichten
import paho.mqtt.client as mqtt  # Voor MQTT-netwerkcommunicatie

# Netwerk-instellingen (moeten overeenkomen met de instellingen in Wokwi)
BROKER   = os.environ.get("IOT_BROKER", "broker.hivemq.com")
POORT    = int(os.environ.get("IOT_POORT", "1883"))
KLASCODE = os.environ.get("IOT_KLASCODE", "i700301-2026-xyz")
TOPIC    = f"ugent/{KLASCODE}/stal/+"

def on_message(client, userdata, msg):
    '''Callback-functie die wordt opgeroepen zodra er een bericht op de broker binnenkomt.'''
    afdeling_naam = msg.topic.split("/")[-1]
    
    # Toon enkel berichten afkomstig van Wokwi-sensoren (die beginnen met 'wokwi-')
    if not afdeling_naam.startswith("wokwi-"):
        return
        
    try:
        b = json.loads(msg.payload.decode())  # Converteer JSON-bytes naar Python dictionary
        
        alarm_tekst = "🚨 ALARM!" if b.get("interrupt") else "OK"
        print(f"[{time.strftime('%H:%M:%S')}] {afdeling_naam:>18s}: "
              f"Temp = {b['temperature']:5.1f} °C | RV = {b['humidity']:5.1f} % | "
              f"Bat = {b['battery']} V | Status: {alarm_tekst}")
    except Exception as fout:
        print(f"Fout bij verwerken van bericht op {msg.topic}: {fout}")

# Initialiseer de MQTT-client
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_message = on_message

# Maak verbinding met de broker en abonneer op het topic
client.connect(BROKER, POORT, keepalive=60)
client.subscribe(TOPIC)

print(f"Luisteren naar live Wokwi-sensoren op topic '{TOPIC}' gedurende 15 seconden...")
client.loop_start()
time.sleep(int(os.environ.get("IOT_LUISTER", "15")))
client.loop_stop()
client.disconnect()
print("Klaar met luisteren.")

Luisteren naar live Wokwi-sensoren op topic 'ugent/i700301-2026-xyz/stal/+' gedurende 15 seconden...


Klaar met luisteren.


<div style="background-color: #e8f5e9; color: #1b5e20; border-left: 5px solid #2e7d32; padding: 12px; margin: 12px 0; border-radius: 4px;">
<strong>💡 Interpretatie van de uitvoer:</strong><br>
<strong>Live Ontvangst van Virtuele Wokwi-sensoren:</strong><br>• <code>[10:20:15]</code>: Het exacte tijdstip waarop het bericht via het internet in dit notebook arriveert.<br>• <code>wokwi-groep01</code>: De unieke identificatienaam van jouw virtuele ESP32 in Wokwi.<br>• <code>Temp = 24.5 °C | RV = 62.0 % | Bat = 3.6 V | Status: OK</code>: De ontleedde JSON-data.<br>Wanneer je in Wokwi de temperatuur boven de 30 °C schuift, verandert het statuslabel hier live in <code>🚨 ALARM!</code>.
</div>

## 5. Verdiepingsopdrachten

1. **Zendinterval en Batterijduur:** Zet in Wokwi de variabele `INTERVAL = 600` (10 minuten), zoals bij een echt commercieel toestel. Wat merk je op bij het testen? Waarom kiezen fabrikanten zoals Dragino voor een interval van 10 tot 20 minuten in plaats van elke seconde?
2. **Wi-Fi Energieverbruik vs LoRaWAN Deep Sleep:** Een Wi-Fi module op een ESP32 verbruikt tijdens het zenden relatief veel stroom (100 tot 200 mA). Hoe beïnvloedt dit de geschiktheid van Wi-Fi voor sensoren die jaren op een knoopcelbatterij moeten werken?
3. **Flexibiliteit van het JSON-dataformaat:** Voeg in Wokwi een extra variabele toe aan het `bericht` dictionary in MicroPython (bijv. `"lichtsterkte": 450`). Kan de verwerkende code in dit notebook het bericht nog steeds correct inlezen zonder vast te lopen? Wat leert dit je over de voordelen van JSON ten opzichte van ruwe binaire hex-bytes?

In [4]:
# ==============================================================================
# STUDENTEN-VERDIEPINGSOPDRACHTEN (WOKWI ESP32 MICROPYTHON)
# ==============================================================================
# SCHRIJF HIER JE ANTWOORDEN / BEVINDINGEN (IN PYTHON COMMENTS OF PRINT-STATEMENTS):
antwoord_opdracht_1 = ""
antwoord_opdracht_2 = ""
antwoord_opdracht_3 = ""


<details>
<summary><b>Oplossing tonen</b></summary>

1. Bij `INTERVAL = 600` wacht je 10 min op de eerste meting. De batterijduur schaalt ongeveer lineair met het interval: zenden (en bij wifi: verbinden) is de grootste verbruiker; Dragino §4 geeft de tabel per interval.
2. Na `machine.deepsleep()` herstart de ESP32 volledig: wifi opnieuw associëren (1–4 s, 100–200 mA) en MQTT opnieuw verbinden. Dat maakt wifi-sensoren op batterij hooguit maanden houdbaar. Bij LoRaWAN blijft de sessie (sleutels, teller) bewaard; een uplink kost enkele tientallen ms zendtijd, vandaar jaren.
3. Zolang je de bestaande sleutels (`temperature`, `humidity`, `time`, …) behoudt en enkel een sleutel toevoegt, leest notebook 02 het bericht nog. Verander je sleutelnamen of de structuur, dan breekt de ontvanger: een **afgesproken berichtformaat** (schema) is de basis van elk IoT-systeem, precies wat de Dragino-handleiding in §2.2 vastlegt.
4. ESP32-ontwikkelbord ± € 10, SHT31-module ± € 10–15 (DHT22 ± € 5), behuizing IP65 ± € 10, USB-voeding ± € 8: ± € 40, maar **netvoeding nodig** (wifi op batterij werkt maanden, geen jaren) en zelf programmeren en onderhouden. Dragino S31-LB € 53 + gateway, batterij tot 10 jaar, kant-en-klaar, maar LoRaWAN-infrastructuur nodig. Zelfbouw wint bij één sensor met stopcontact en wifi in bereik; Dragino wint zodra je meerdere sensoren zonder stroom en zonder onderhoud wilt.

</details>